# Projeto Original — Fine-tuning com RoBERTa-base

**Ciclo de divergência: `MODEL` (abordagem de alto desempenho).**

Os notebooks anteriores compararam representações textuais (TF-IDF, BM25, BGE-large, BGE-M3) usando os embeddings **congelados** + classificador linear. Aqui mudamos o paradigma: fazemos o **fine-tuning** de um transformer especializado em classificação, o **RoBERTa-base**, que é o tipo de modelo que domina o estado da arte do IMDB (~96%).

Mantemos os mesmos dados e a mesma divisão treino/teste dos outros notebooks, para que o F1 macro seja comparável. A diferença é que agora **todos os pesos da rede são ajustados à tarefa**, em vez de apenas extrair vetores fixos.

## Dependências

In [10]:
!pip install -q transformers accelerate scikit-learn imbalanced-learn pandas

In [11]:
import os
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

Device: cuda


## Configuração

`N_PER_CLASS` controla a escala (10000 reproduz o run de 20k). `MAX_LEN` é o limite de tokens (RoBERTa aceita até 512; 256 é mais rápido e suficiente para a maioria das críticas — aumente para 512 se quiser o máximo de desempenho).

In [12]:
MODEL_NAME = 'roberta-base'
RANDOM_STATE = 42
N_PER_CLASS = 10000   # 10000/10000 = 20.000 registros
MAX_LEN = 512         # aumente para 512 para o melhor desempenho (mais lento)
EPOCHS = 3
BATCH = 8
LR = 2e-5

## Dados

Faça o upload do arquivo `IMDB Dataset.csv` quando solicitado (ou coloque-o em `/content`).

In [13]:
candidates = ['IMDB Dataset.csv', '/content/IMDB Dataset.csv', '../IMDB Dataset.csv']
csv_path = next((p for p in candidates if os.path.exists(p)), None)

if csv_path is None:
    try:
        from google.colab import files
        print('Faça upload do IMDB Dataset.csv:')
        uploaded = files.upload()
        csv_path = list(uploaded.keys())[0]
    except Exception:
        raise FileNotFoundError('IMDB Dataset.csv não encontrado. Faça upload para /content.')

df_review = pd.read_csv(csv_path)
print('Total de linhas no CSV:', len(df_review))
df_review.head()

Total de linhas no CSV: 50000


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [14]:
# Sanidade: confirma que o CSV veio inteiro (pega upload truncado).
assert len(df_review) == 50000, f'Esperado 50000 linhas, veio {len(df_review)} — arquivo truncado?'

# Mesma amostragem do notebook "Original, 10k dados" para comparabilidade direta:
# remove reviews duplicadas (evita vazamento treino/teste) e tira amostra
# balanceada aleatória e reprodutível (mesmo RANDOM_STATE).
df_dedup = df_review.drop_duplicates(subset=['review'])
df_review_bal = (
    df_dedup.groupby('sentiment', group_keys=False)
    .sample(n=N_PER_CLASS, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)
print(df_review_bal.value_counts('sentiment'))

# Mesma divisão treino/teste (mesmo random_state, sem stratify) -> mesmo conjunto de teste.
train_df, test_df = train_test_split(
    df_review_bal, test_size=0.33, random_state=RANDOM_STATE
)
# Rótulos numéricos para o RoBERTa (negative=0, positive=1).
train_df = train_df.assign(label=(train_df['sentiment'] == 'positive').astype(int))
test_df = test_df.assign(label=(test_df['sentiment'] == 'positive').astype(int))

print('Treino:', len(train_df), '| Teste:', len(test_df))
print(train_df['sentiment'].value_counts())

sentiment
negative    10000
positive    10000
Name: count, dtype: int64
Treino: 13400 | Teste: 6600
sentiment
positive    6729
negative    6671
Name: count, dtype: int64


## Tokenização

In [15]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_enc = tokenizer(list(train_df['review']), truncation=True, max_length=MAX_LEN)
test_enc = tokenizer(list(test_df['review']), truncation=True, max_length=MAX_LEN)

class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_ds = IMDbDataset(train_enc, train_df['label'])
test_ds = IMDbDataset(test_enc, test_df['label'])

## Modelo e Trainer

In [16]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro'),
    }

args = TrainingArguments(
    output_dir='roberta_imdb',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=32,
    learning_rate=LR,
    fp16=(DEVICE == 'cuda'),
    logging_steps=50,
    report_to='none',
)

collator = DataCollatorWithPadding(tokenizer)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Treinamento (fine-tuning)

In [17]:
trainer.train()

Step,Training Loss
50,0.638936
100,0.415628
150,0.379868
200,0.408450
250,0.311644
300,0.411674
350,0.319080
400,0.245135
450,0.355757
500,0.366665


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5025, training_loss=0.18920488295863516, metrics={'train_runtime': 1268.2163, 'train_samples_per_second': 31.698, 'train_steps_per_second': 3.962, 'total_flos': 9808743143828640.0, 'train_loss': 0.18920488295863516, 'epoch': 3.0})

## Avaliação

In [18]:
pred = trainer.predict(test_ds)
y_pred = np.argmax(pred.predictions, axis=-1)
y_true = test_df['label'].values

acc = accuracy_score(y_true, y_pred)
f1m = f1_score(y_true, y_pred, average='macro')

print('=== RoBERTa-base (fine-tuning) ===')
print(f'Acurácia: {acc:.4f} | F1 macro: {f1m:.4f}')
print()
print(classification_report(y_true, y_pred, target_names=['negative', 'positive']))
print('Matriz de confusão (linhas=real, colunas=previsto) [negative, positive]:')
print(confusion_matrix(y_true, y_pred))

=== RoBERTa-base (fine-tuning) ===
Acurácia: 0.9505 | F1 macro: 0.9505

              precision    recall  f1-score   support

    negative       0.96      0.95      0.95      3329
    positive       0.94      0.96      0.95      3271

    accuracy                           0.95      6600
   macro avg       0.95      0.95      0.95      6600
weighted avg       0.95      0.95      0.95      6600

Matriz de confusão (linhas=real, colunas=previsto) [negative, positive]:
[[3146  183]
 [ 144 3127]]


## Comparação

Insira o `F1 macro` obtido acima na tabela comparativa do README, como a abordagem de **fine-tuning** — distinta das representações congeladas:

| Abordagem | Melhor F1 macro (20k) |
|---|---|
| TF-IDF | ~0,876 |
| BM25 | ~0,877 |
| BGE-M3 (congelado) | ~0,920 |
| BGE-large (congelado) | ~0,944 |
| **RoBERTa-base (fine-tuning)** | **(preencher)** |

Esperado: 0,95–0,96, aproximando-se do estado da arte do IMDB (0,96–0,97 de acurácia).